In [3]:
#### create a CHiCAGO consensus between fragment and 5kb resolutions
### add ABC fragment resolution (mapped to fragment rmap)
### remove redundant interactions. Priority: fragment CHiCAGO, "5kb" CHiCAGO, ABC 5kb
### this consensus is used for running COGS and multiCOGS

library(data.table)
# library(Chicago)

### loading ILC Chicago and ABC

ilc_fres <- readRDS("~/miniPCHiC/hILCs/hILCs_run2/Chicago/without_binning/hILC3_all_merged_Step2_corrected_weighting/data/hILC3_all_merged_Step2_corrected_weighting.Rds")
ilc_5kb <- readRDS("~/miniPCHiC/hILCs/hILCs_run2/Chicago/with_binning_5kb_solbaits/hILCs_all_merged_bin_Step2/data/hILCs_all_merged_bin_Step2.Rds")
ilc_abc_5kb <- fread("/rds/general/user/malyshev/home/spivakov_analysis_live/artemov/ABC_MASTER/ABC_data/output/ILC_old_activity_bin/EnhancerPredictionsAllPutative.txt.gz")

### loading CD4 Chicago and ABC

# cd4_fres <- readRDS("~/miniPCHiC/PCHiC_DpnII_merged/Chicago/without_binning/M1_merged_Step2_reweighting/data/M1_merged_Step2_reweighting.Rds")
# cd4_5kb <- readRDS("~/miniPCHiC/PCHiC_DpnII_merged/Chicago/with_binning/M1_bin5K_merged_Step2/data/M1_bin5K_merged_Step2.Rds")
# cd4_abc_5kb <- fread("/rds/general/user/malyshev/home/spivakov_analysis_live/artemov/ABC_MASTER/ABC_data/output/CD4_newact_bin/EnhancerPredictionsAllPutative.txt.gz")


In [4]:
head(ilc_abc_5kb)

chr,start,end,name,class,activity_base,TargetGene,TargetGeneTSS,TargetGeneExpression,TargetGenePromoterActivityQuantile,⋯,powerlaw_contact_reference,hic_contact,hic_contact_pl_scaled,hic_pseudocount,hic_contact_pl_scaled_adj,ABC.Score.Numerator,ABC.Score,powerlaw.Score.Numerator,powerlaw.Score,CellType
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<int>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,DUSP22,292056,1670,0.914700,⋯,0.006398,0.864560,0.864560,0.000878,0.864560,0.164680,0.002143,0.001219,0.009426,ILC_old_activity_bin
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,IRF4,391738,408,0.563648,⋯,0.003731,0.594669,0.594669,0.000878,0.594669,0.113271,0.001922,0.000711,0.003696,ILC_old_activity_bin
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,HUS1B,656964,6,0.277060,⋯,0.001754,0.374767,0.374767,0.000878,0.374767,0.071385,0.001421,0.000334,0.001196,ILC_old_activity_bin
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,EXOC2,693141,1660,0.722715,⋯,0.001635,0.360310,0.360310,0.000878,0.360310,0.068631,0.000183,0.000311,0.000273,ILC_old_activity_bin
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,FOXCUT,1605530,NaN,0.459661,⋯,0.000597,0.220015,0.220015,0.000597,0.220015,0.041908,0.000324,0.000114,0.000219,ILC_old_activity_bin
chr6,148043,148543,promoter|chr6:148043-148543,promoter,0.190478,FOXC1,1610445,35,0.337191,⋯,0.000595,0.219694,0.219694,0.000595,0.219694,0.041847,0.000480,0.000113,0.000259,ILC_old_activity_bin


In [3]:
### merge fragment and 5kb rmaps by midpoints

rmap_merge <- function(chicago_fres, chicago_5kb){
    
    ## loading rmaps
    rmap_fres <- fread(chicago_fres@settings$rmapfile)
    rmap_5kb <- fread(chicago_5kb@settings$rmapfile)
    
    # find fragment midpoints
    rmap_fres[,mid1 := (V2+V3)/2][,mid2 := mid1] 
    
    setkey(rmap_fres, V1, mid1, mid2)
    setkey(rmap_5kb, V1, V2, V3)
    
    ## assign fragment-res IDs to 5kb-res IDs by the  of fragment-res
    rmap_fres_5kb <- foverlaps(rmap_fres, rmap_5kb, by.x = c("V1", "mid1", "mid2"), 
                               by.y = c("V1", "V2", "V3"), type = "within") 
    
    colnames(rmap_fres_5kb) <- c("chr", "start_5kb", "end_5kb", "fragID_5kb", 
                                 "start_fres", "end_fres", "fragID_fres", "mid1", "mid2")
    
    return(rmap_fres_5kb[,c("chr", "start_5kb", "end_5kb", "fragID_5kb", 
                                 "start_fres", "end_fres", "fragID_fres")])

}

rmap_fres_5kb <- rmap_merge(ilc_fres, ilc_5kb) ## the result is the same for the cd4_fres/cd_5kb
# rmap_fres_5kb <- rmap_merge(cd4_fres, cd4_5kb) ## the result is the same for the cd4_fres/cd_5kb


In [140]:
### assign fres contact IDs to 5kb contact IDs in CHiCAGO

chicago_res_convert <- function(chicago_5kb, chicago_fres, chicago_cutoff, rmap_merged){
    
    ## selecting significanty interactions; removing interchromosomal interactions
    chicago_5kb_sel <- chicago_5kb@x[score>=chicago_cutoff][!is.na(distSign)] 
    
    ## annotating 5kb baitIDs as fragment baitIDs
    chicago_5kb_fres <- merge(chicago_5kb_sel, 
                            rmap_merged[,c("fragID_5kb", "fragID_fres", "chr", "start_fres", "end_fres")], 
                            by.x = c("baitID"), by.y = c("fragID_5kb"), all.x = TRUE) 
    
    ## renaming old baitID as baitID_5kb and new baitID_fres as baitID
    setnames(chicago_5kb_fres, 
             c("baitID", "fragID_fres", "chr", "start_fres", "end_fres", "score", "N"), 
             c("baitID_5kb", "baitID_fres", "baitChr", "bait_start_fres", "bait_end_fres", "chicago_score_5kb", "N_5kb")) 

    ## annotating 5kb oeIDs as fragment oeIDs
    chicago_5kb_fres <- merge(chicago_5kb_fres, 
                          rmap_merged[,c("fragID_5kb", "fragID_fres", "chr", "start_fres", "end_fres")],
                          by.x = c("otherEndID"), by.y = c("fragID_5kb"), all.x = TRUE) 

    ## renaming old otherEndID as oeID_5kb and new baitID_fres as oeID
    setnames(chicago_5kb_fres, 
             c("otherEndID", "fragID_fres", "chr", "start_fres", "end_fres"), 
             c("oeID_5kb", "oeID_fres", "oeChr", "oe_start_fres", "oe_end_fres")) 
    
    ## load and format baitmap at fragment resolution
    bmap_fres <- fread(chicago_fres@settings$baitmapfile)
    colnames(bmap_fres) <- c("baitChr", "baitStart", "baitEnd", "baitID", "Target")

    ## annotating baitID target as per fragment bmap
    chicago_5kb_fres <- merge(chicago_5kb_fres, bmap_fres[,c("baitID", "Target")], by.x = "baitID_fres", by.y = "baitID", all.x = TRUE) 
    setnames(chicago_5kb_fres, "Target", "baitName") 
    
    ## selecting columns for the peakmatrix
    chicago_5kb_fres <- chicago_5kb_fres[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", 
                                            "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres",
                                            "baitID_5kb", "oeID_5kb", "chicago_score_5kb", "N_5kb")]   
    
    ## creaing contactID, which will be used for filtering redundant interactions
    chicago_5kb_fres[,fres_contactID := paste0(baitID_fres, "_", oeID_fres)] 
    
    return(chicago_5kb_fres)

}

# ## for ILCs
ilc_5kb_fres <- chicago_res_convert(chicago_5kb = ilc_5kb, chicago_fres = ilc_fres, chicago_cutoff = 5, rmap_merged = rmap_fres_5kb)

## for CD4+ T cells
# cd4_5kb_fres <- chicago_res_convert(chicago_5kb = cd4_5kb, chicago_fres = cd4_fres, chicago_cutoff = 5, rmap_merged = rmap_fres_5kb)


In [142]:
### selecting chicago interactions at fragment resolution

chicago_fres_reform <- function(chicago_fres, chicago_cutoff){
    
    ## keeping only significanty interactions; removing interchromosomal interactions
    chicago_fres_sel <- chicago_fres@x[score>=chicago_cutoff][!is.na(distSign)]  
    
    ## selecting rmap
    rmap_fres <- fread(chicago_fres@settings$rmapfile)
    colnames(rmap_fres) <- c("chr", "start", "end", "ID")
    
    ## annotate baitID coordinates in chicago fragres interactions
    chicago_fres_sel <- merge(chicago_fres_sel, rmap_fres[,c("chr", "start", "end", "ID")], by.x = c("baitID"), by.y = c("ID"), all.x = TRUE)
    
    setnames(chicago_fres_sel, 
             c("chr", "baitID", "start", "end", "score", "N"), 
             c("baitChr", "baitID_fres", "bait_start_fres", "bait_end_fres", "chicago_score_fres", "N_fres"))
    
    ## annotate otherEndID coordinates in chicago fragres interactions
    chicago_fres_sel <- merge(chicago_fres_sel, rmap_fres[,c("chr", "start", "end", "ID")], by.x = c("otherEndID"), by.y = c("ID"), all.x = TRUE)
    setnames(chicago_fres_sel, 
             c("chr", "otherEndID", "start", "end"), 
             c("oeChr", "oeID_fres", "oe_start_fres", "oe_end_fres"))
    
    ## load and format baitmap at fragment resolution
    bmap_fres <- fread(chicago_fres@settings$baitmapfile)
    colnames(bmap_fres) <- c("baitChr", "baitStart", "baitEnd", "baitID", "Target")
    
    ## annotate targets in baitIDs
    chicago_fres_sel <- merge(chicago_fres_sel, bmap_fres[,c("baitID", "Target")], by.x = "baitID_fres", by.y = "baitID", all.x = TRUE)
    setnames(chicago_fres_sel, "Target", "baitName")

    ## selecting columns for the peakmatrix
    chicago_fres_sel <- chicago_fres_sel[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", 
                                            "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "chicago_score_fres", "N_fres")]
    
    ## creaing contactID, which will be used for filtering redundant interactions
    chicago_fres_sel[,fres_contactID := paste0(baitID_fres, "_", oeID_fres)]

    return(chicago_fres_sel)

}

# ## for ILCs
ilc_fres_sel <- chicago_fres_reform(chicago_fres = ilc_fres, chicago_cutoff = 5)

## for CD4+ T cells
# cd4_fres_sel <- chicago_fres_reform(chicago_fres = cd4_fres, chicago_cutoff = 5)


In [144]:
### assign ABC promoter-enhancer pairs to rmap_5kb

abc_prepare <- function(abc_file, foverlaps_type, rmap_merged, ABC_cutoff){
  

    ## formatting abc results
    
    print("formatting abc results")
    
    # 'chr' formatting; only keeping enhancers, which do not overlap promoter regions 
    abc_file <- abc_file[,chr := gsub("chr", "", chr)][class != "promoter"][ABC.Score >= ABC_cutoff]
    setnames(abc_file, c("chr", "start", "end"), c("chr", "peak_start", "peak_end"))

    ## overlap ABC regions with rmap
    # foverlaps type : "any" or "within"

    setkey(rmap_merged, chr, start_5kb, end_5kb)
    
    ## overlapping TSS with rmap

    print("overlapping TSS with rmap")
    
    abc_file[,TSS_2 := TargetGeneTSS]
    
    setkey(abc_file, chr, TargetGeneTSS, TSS_2)
       
    abc_file <- foverlaps(abc_file, 
                         rmap_merged, 
                         by.x = c("chr", "TargetGeneTSS", "TSS_2"), 
                         by.y = c("chr", "start_5kb", "end_5kb"), 
                         type = "within")
    
    setnames(abc_file, 
                 c("chr", "start_5kb", "end_5kb", "fragID_5kb", "start_fres", "end_fres", "fragID_fres"), 
                 c("baitChr", "bait_start_5kb",  "bait_end_5kb", "baitID_5kb", "bait_start_fres", "bait_end_fres", "baitID_fres"))

    
    ## overlapping regions with rmap
    
    print("overlapping regions with rmap")
    
    if (foverlaps_type == "any"){
        
        setkey(abc_file, baitChr, peak_start, peak_end)
        
        abc_all <- foverlaps(abc_file, 
                             rmap_merged, 
                             by.x = c("baitChr", "peak_start", "peak_end"), 
                             by.y = c("chr", "start_5kb", "end_5kb"), 
                             type = "any")
        
        setnames(abc_all, 
                 c("start_5kb", "end_5kb", "fragID_5kb", "start_fres", "end_fres", "fragID_fres"), 
                 c("oe_start_5kb",  "oe_end_5kb", "oeID_5kb", "oe_start_fres", "oe_end_fres", "oeID_fres"))
        
        
    } else {
        abc_file[mid := (peak_start + peak_end) /2 ][mid2 := mid] 
        
        setkey(abc_file, chr, mid, mid2)

        abc_all <- foverlaps(abc_file, 
                             rmap_merged, 
                             by.x = c("chr", "mid", "mid2"), 
                             by.y = c("chr", "start_5kb", "end_5kb"), 
                             type = foverlaps_type)
        
        setnames(abc_all, 
                 c("start_5kb", "end_5kb", "fragID_5kb", "start_fres", "end_fres", "fragID_fres"), 
                 c("oe_start_5kb",  "oe_end_5kb", "oeID_5kb", "oe_start_fres", "oe_end_fres", "oeID_fres"))
        
    }

    print("compressing the data")
       
    ## selecting
    
    abc_all <- unique(abc_all[baitID_5kb != oeID_5kb][class != "promoter"][, c("baitID_5kb", "oeID_5kb",  'baitChr', 'bait_start_fres', 'bait_end_fres', 
                                                                               'baitID_fres', 'oe_start_fres', 'oe_end_fres', 'oeID_fres', "hic_contact_pl_scaled_adj", 
                                                                               "ABC.Score", "TargetGene", "ABC.Score.Numerator")])
#     ## consider:
#     abc_all <- abc_all[baitID_5kb != oeID_5kb][class != "promoter"][, c("baitID_5kb", "oeID_5kb",  'baitChr', 'bait_start_fres', 'bait_end_fres', 
#                                                                                'baitID_fres', 'oe_start_fres', 'oe_end_fres', 'oeID_fres', "hic_contact_pl_scaled_adj", 
#                                                                                "ABC.Score", "TargetGene", "ABC.Score.Numerator")]
    
    
    abc_all <- abc_all[,list(ABC.Score = max(ABC.Score), 
                             baitID_5kb = unique(baitID_5kb),
                             oeID_5kb = unique(oeID_5kb),
                             ABC.Score.Numerator = max(ABC.Score.Numerator), 
                             N_abc = max(hic_contact_pl_scaled_adj), 
                             TargetGene = paste0(TargetGene, collapse = ",")), 
                       by = c('baitChr', "bait_start_fres", "bait_end_fres", "baitID_fres", 'oe_start_fres', 'oe_end_fres', 'oeID_fres')]
    
    setnames(abc_all, "TargetGene", "baitName")
    
    abc_all <- abc_all[,c("baitID_5kb", "oeID_5kb", 'baitChr', "bait_start_fres", "bait_end_fres", "baitID_fres", 'oe_start_fres', 'oe_end_fres', 'oeID_fres', 
                          "N_abc", "ABC.Score", "ABC.Score.Numerator")]
  
    return(abc_all)
}

# ### preparing ABC calls for ILC3s
ilc_abc_prepared <- abc_prepare(abc_file = ilc_abc_5kb, foverlaps_type = "any", rmap_merged = rmap_fres_5kb, ABC_cutoff = 0.023)

### preparing ABC calls for CD4+ T cells
# cd4_abc_prepared <- abc_prepare(abc_file = cd4_abc_5kb, foverlaps_type = "any", rmap_merged = rmap_fres_5kb, ABC_cutoff = 0.023)


[1] "formatting abc results"
[1] "overlapping TSS with rmap"
[1] "overlapping regions with rmap"
[1] "compressing the data"


In [95]:
### sanity check

cd4_abc_prepared[,full_bait_ID := paste(bait_start_fres, bait_end_fres, baitID_fres, sep = "_")]
cd4_abc_prepared[,full_oe_ID := paste(oe_start_fres, oe_end_fres, oeID_fres, sep = "_")]

rmap <- fread("~/spivakov_analysis_live/Design/Human_DpnII_binsize1500_maxL75K/hg38_dpnII.rmap")
rmap[, full_frag_ID := paste(V2, V3, V4, sep = "_")]

any(!(cd4_abc_prepared$full_oe_ID %in% rmap$full_frag_ID))
any(!(cd4_abc_prepared$full_bait_ID %in% rmap$full_frag_ID))

In [145]:
### merge chicago fres and 5kb resolution interactions

chicago_merge <- function(chicago_fres_sel, # selected chicago interactions at fragment resolution
                          chicago_5kb_fres) # chicago 5kb interactions annotated at fragment resolution
{
    # merging by fragment resolution baitIDs and oeIDs and their corresponding info
    peakm_chic <- merge(chicago_fres_sel, 
                        chicago_5kb_fres, 
                        by = c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", 
                               "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "fres_contactID"), 
                        all = TRUE)
    
    # baitName re-assignement
    peakm_chic[,baitName := ifelse(is.na(baitName.x), baitName.y, baitName.x)]
    peakm_chic[,c("baitName.x", "baitName.y") := NULL]
#     peakm_chic[,oeName := ifelse(is.na(oeName.x), oeName.y, oeName.x)]
#     peakm_chic[,c("oeName.x", "oeName.y") := NULL]
    
#     # distance re-assignement
#     peakm_chic[,dist := ifelse(is.na(dist.x), dist.y, dist.x)]
#     peakm_chic[,c("dist.x", "dist.y") := NULL]
#     peakm_chic[,N := ifelse(is.na(N.x), N.y, N.x)]
#     peakm_chic[,c("N.x", "N.y") := NULL]
    
    return(peakm_chic)

}

# # merging peakmatrices for ILCs
ilc_peakm_chic <- chicago_merge(chicago_fres_sel = ilc_fres_sel, chicago_5kb_fres = ilc_5kb_fres)

# merging peakmatrices for CD4+ T cells
# cd4_peakm_chic <- chicago_merge(chicago_fres_sel = cd4_fres_sel, chicago_5kb_fres = cd4_5kb_fres)


In [147]:
## merge ABC with chicago merged peakmatrix
## note the ABC cutoff can vary across the cell types and resolutions

abc_chic_merge <- function(peakm_chic, abc_prepared, chicago_cutoff, ABC_cutoff){
    
    peakm_chic_abc <- merge(peakm_chic, 
                            abc_prepared, 
                            by = c('baitID_5kb','oeID_5kb', 'baitChr','bait_start_fres','bait_end_fres','baitID_fres','oe_start_fres','oe_end_fres','oeID_fres'), 
                            all = TRUE)
    
#     # formatting the target names 
#     peakm_chic_abc[,baitName := ifelse(is.na(baitName.x), baitName.y, baitName.x)]
#     peakm_chic_abc[,c("baitName.x", "baitName.y") := NULL]
#     peakm_chic_abc[,oeName := ifelse(is.na(oeName.x), oeName.y, oeName.x)]
#     peakm_chic_abc[,c("oeName.x", "oeName.y") := NULL]
    
#     # only retaining the intrachromosomal interactions
#     peakm_chic_abc <- peakm_chic_abc[baitChr == oeChr]
    
#     # formatting the counts
#     peakm_chic_abc[,N := ifelse(is.na(N.x), N.y, N.x)]
#     peakm_chic_abc[,c("N.x", "N.y") := NULL]
    
    # formatting the distance
    peakm_chic_abc[,dist := abs((oe_start_fres + oe_end_fres)/2- (bait_start_fres + bait_end_fres)/2)]

    # formatting the ABC.Score
    ## if higher than the ABC_threshold make it as if it is "chicago score" of greater than 5 (significant)
    ## this is to avoid loosing ABC interactions accidentally during any further filtering by score
    
    peakm_chic_abc[,ABC.Score := ifelse(ABC.Score >= ABC_cutoff, chicago_cutoff, ABC.Score)]
    
    ## selecting columns to retain
    peakm_chic_abc <- peakm_chic_abc[,c("baitChr", "baitStart", "baitEnd", "baitID", "baitName", "oeChr", "oeStart", "oeEnd", "oeID", "oeName", "N", "baitID_5kb", "oeID_5kb", "dist", "chicago_fres", "chicago_5kb", "ABC.Score", "ABC.Score.Numerator")]
    
    return(peakm_chic_abc)
}

# ## merge for ILC3s on revised ABC cut-off (0.023)
ilc_abc_chic_peakm <- abc_chic_merge(peakm_chic = ilc_peakm_chic, abc_prepared = ilc_abc_prepared, chicago_cutoff = 5, ABC_cutoff = 0.023)

## merge for CD4+ T cells on revised ABC cut-off (0.023)
# cd4_abc_chic_peakm <- abc_chic_merge(peakm_chic = cd4_peakm_chic, abc_prepared = cd4_abc_prepared, chicago_cutoff = 5, ABC_cutoff = 0.023)


In [149]:
## summary of all interactions

count_all <- function(chicago_fres, chicago_5kb, chicago_cutoff, abc_prepared, abc_cutoff, peakm_chic_abc){
    
    print(paste0("total number of unique interactions in the chicago object at fragment resolution: ", nrow(chicago_fres@x[score >= chicago_cutoff][!is.na(distSign)])))
    
    print(paste0("total number of unique interactions in the chicago object at 5kb resolution: ", nrow(chicago_5kb@x[score >= chicago_cutoff][!is.na(distSign)])))
    
    print(paste0("total number of ABC predicted elements in the ABC file mapped to rmap: ", nrow(abc_prepared[ABC.Score > abc_cutoff])))
        
    print(paste0("total number of unique signficant interactions in the merged peakmatrix at fragment resolution: ", nrow(unique(peakm_chic_abc[chicago_score_fres >= chicago_cutoff][,c("baitID_fres", "oeID_fres", "chicago_score_fres")]))))
    
    print(paste0("total number of unique signficant interactions in the merged peakmatrix at 5kb resolution: ", nrow(unique(peakm_chic_abc[chicago_score_5kb >= chicago_cutoff][,c("baitID_5kb", "oeID_5kb", "chicago_score_5kb")]))))
    
    print(paste0("total number of unique ABC-predicted regulatory elements in the merged peakmatrix: ", nrow(peakm_chic_abc[ABC.Score >= abc_cutoff])))

}

# ## ILCs summary
count_all(ilc_fres, ilc_5kb, 5 , ilc_abc_prepared, 0.023,  ilc_abc_chic_peakm)

## CD4 summary
# count_all(cd4_fres, cd4_5kb, 5, cd4_abc_prepared, 0.023, cd4_abc_chic_peakm)


[1] "total number of unique interactions in the chicago object at fragment resolution: 27600"
[1] "total number of unique interactions in the chicago object at 5kb resolution: 58235"
[1] "total number of ABC predicted elements in the ABC file mapped to rmap: 1027982"
[1] "total number of unique signficant interactions in the merged peakmatrix at fragment resolution: 27600"
[1] "total number of unique signficant interactions in the merged peakmatrix at 5kb resolution: 58235"
[1] "total number of unique ABC-predicted regulatory elements in the merged peakmatrix: 1027982"


In [150]:
## need to adjust from here on

fwrite(ilc_abc_chic_peakm[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "baitID_5kb", "oeID_5kb", "dist", "N_fres", "N_5kb", "N_abc", "chicago_score_fres", "chicago_score_5kb", "ABC.Score")],
       "~/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025.txt", sep = "\t", scipen = 999)

# fwrite(cd4_abc_chic_peakm[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "baitID_5kb", "oeID_5kb", "dist", "N_fres", "N_5kb", "N_abc", "chicago_score_fres", "chicago_score_5kb", "ABC.Score")],
#         "~/miniPCHiC/PCHiC_DpnII_merged/CD4_chicago_fres_5kb_abc_023_fres_extended_peakm_13012025.txt", sep = "\t", scipen = 999)


In [152]:
consensus_peakmatrix <- function(peakm_chic_abc, chicago_cutoff, abc_cutoff){
    
    redundant_5kb <- unique(peakm_chic_abc[chicago_score_fres >= chicago_cutoff & chicago_score_5kb >= chicago_cutoff][, contactID_5kb := paste(baitID_5kb, oeID_5kb, sep = "_")]$contactID_5kb)

    peakm_chic_abc[,contactID_5kb := paste(baitID_5kb, oeID_5kb, sep = "_")]
    
    ### below I am annotating interactions "redundant", which are not called at fragment resolution, 
    ### but fall within a 5kb interaction, which
    ### contains a signficant fragment resolution contact, different from the one being redundant
    
    peakm_chic_abc[,remove_contact := ifelse(((contactID_5kb %in% redundant_5kb) & (chicago_score_fres < chicago_cutoff | is.na(chicago_score_fres))), "TRUE",  "FALSE")]

    peakm_chic_abc_sel <- peakm_chic_abc[remove_contact == FALSE]
    peakm_chic_abc_sel[,c("contactID_5kb", "remove_contact") := NULL]

        
    print(paste0("total number of unique signficant interactions in the merged peakmatrix at fragment resolution: ", nrow(unique(peakm_chic_abc_sel[chicago_score_fres >= chicago_cutoff][,c("baitID_fres", "oeID_fres", "chicago_score_fres")]))))
    
    print(paste0("total number of unique signficant interactions in the merged peakmatrix at 5kb resolution: ", nrow(unique(peakm_chic_abc_sel[chicago_score_5kb >= chicago_cutoff][,c("baitID_5kb", "oeID_5kb", "chicago_score_5kb")]))))

    ### all ABC-called regions that are removed, are removed only because ABC was called at 5kb resolution and therefore all fragment resolution enhancers were called "signficant"
    ### therefore a fragment resolution chicago call takes priority over the ABC here.
    print(paste0("total number of unique ABC-predicted regulatory elements in the merged peakmatrix: ", nrow(peakm_chic_abc_sel[ABC.Score >= abc_cutoff])))

    return(peakm_chic_abc_sel)

}

ilc_peakm_chic_abc_sel <- consensus_peakmatrix(ilc_abc_chic_peakm, chicago_cutoff = 5, abc_cutoff = 0.023)
# cd4_peakm_chic_abc_sel <- consensus_peakmatrix(cd4_abc_chic_peakm, chicago_cutoff = 5, abc_cutoff = 0.023)

[1] "total number of unique signficant interactions in the merged peakmatrix at fragment resolution: 27600"
[1] "total number of unique signficant interactions in the merged peakmatrix at 5kb resolution: 58235"
[1] "total number of unique ABC-predicted regulatory elements in the merged peakmatrix: 1019337"


In [153]:
fwrite(ilc_peakm_chic_abc_sel[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "baitID_5kb", "oeID_5kb", "dist", "N_fres", "N_5kb", "N_abc", "chicago_score_fres", "chicago_score_5kb", "ABC.Score")],
       "~/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_5kb_abc_023_fres_consensus_peakm_13012025.txt", sep = "\t", scipen = 999)

# fwrite(cd4_peakm_chic_abc_sel[,c("baitChr", "bait_start_fres", "bait_end_fres", "baitID_fres", "baitName", "oeChr", "oe_start_fres", "oe_end_fres", "oeID_fres", "baitID_5kb", "oeID_5kb", "dist", "N_fres", "N_5kb", "N_abc", "chicago_score_fres", "chicago_score_5kb", "ABC.Score")],
#        "~/miniPCHiC/PCHiC_DpnII_merged/CD4_chicago_fres_5kb_abc_023_fres_consensus_peakm_13012025.txt", sep = "\t", scipen = 999)


In [157]:
### sanity check

ilc3_extended <- fread("~/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025.txt")
cd4_extended <- fread("~/miniPCHiC/PCHiC_DpnII_merged/CD4_chicago_fres_5kb_abc_023_fres_extended_peakm_13012025.txt")
rmap <- fread("~/spivakov_analysis_live/Design/Human_DpnII_binsize1500_maxL75K/hg38_dpnII.rmap")

rmap[, full_frag_ID := paste(V2, V3, V4, sep = "_")]

ilc3_extended[, full_bait_ID := paste(bait_start_fres, bait_end_fres, baitID_fres, sep = "_")]
ilc3_extended[, full_oe_ID := paste(oe_start_fres, oe_end_fres, oeID_fres, sep = "_")]

any(!(ilc3_extended$full_bait_ID %in% rmap$full_frag_ID))
any(!(ilc3_extended$full_oe_ID %in% rmap$full_frag_ID))


cd4_extended[, full_bait_ID := paste(bait_start_fres, bait_end_fres, baitID_fres, sep = "_")]
cd4_extended[, full_oe_ID := paste(oe_start_fres, oe_end_fres, oeID_fres, sep = "_")]


any(!(cd4_extended$full_bait_ID %in% rmap$full_frag_ID))
any(!(cd4_extended$full_oe_ID %in% rmap$full_frag_ID))

[1] FALSE

[1] FALSE

[1] FALSE

[1] FALSE